# AgriQDL — Full Training Pipeline

**To register a NEW plantation:** edit the `SITE_NAME`, `SITE_LAT`, `SITE_LON` values near the top of **Cell 1** below (the AgriQDL app's "Add Site" screen gives you these exact values to paste in), then run every cell below in order, top to bottom.

**Total time: roughly 1.5-2.5 hours**, mostly Cell 4 (training). You can close this tab while it runs — Colab keeps working in the background as long as you don't close the whole browser or let it sit idle too long.

| Cell | What it does |
|---|---|
| 1 | Fetch real climate + soil data for this site |
| 2 | Turn raw data into model-ready features |
| 3 | Define the 3 model architectures |
| 4 | Train all 3 models (the long step) |
| 5 | Evaluate with 6 tests, save results |
| 6 | Export weights for the app |


## Cell 1 — Data Pipeline
Fetches NASA POWER (climate) and FAO SoilGrids (soil) data for this site.

In [ ]:
"""
AgriQDL — FINAL COMPREHENSIVE SCRIPT
================================================================================
One authoritative, self-contained run covering:
  - Data pipeline (NASA POWER + FAO SoilGrids)
  - Feature engineering + target generation
  - THREE models trained in one session: AgriQDL, CNN-LSTM baseline,
    standalone CNN baseline (all fairly timed against each other)
  - SIX tests: RMSE, MAE, R² (per target, all 3 models), parameter count,
    training time, paired t-tests (3 pairwise comparisons)
  - Farmer advisory generation with quantum uncertainty sampling

DEFAULT SITE: Sekinchan Paddy Field, Selangor. To run this for a DIFFERENT
plantation, change ONLY the four SITE_* values in the CONFIG section below,
then run this whole script top to bottom -- everything else (data fetch,
training, evaluation, advisory) adapts automatically. This is exactly what
the AgriQDL app's "Add New Plantation" screen prepares for you.

HOW TO RUN:
  Paste this entire file into ONE Colab cell and run it. Total time:
  roughly 1-1.5 hours (mostly AgriQDL + CNN-LSTM + CNN training). Fetches
  fresh climate/soil data, so no need to have run anything before this.
"""

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "pandas", "numpy", "torch", "pennylane", "scipy"], check=True)

import math, time, json, datetime
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy import stats
import pennylane as qml

# =============================================================================
# CONFIG -- change ONLY these 4 values to run this for a different plantation
# =============================================================================
SITE_NAME = "Sekinchan Paddy Field, Jalan Parit 6, Sekinchan, Selangor"
SITE_LAT = 3.5294
SITE_LON = 101.123961
SITE_CROP = "paddy rice"

SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
SCALE_PENALTY_WEIGHT = 1e-4
GRAD_CLIP_NORM = 5.0
FULL_MAX_EPOCHS = 200
BASELINE_MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 15

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Site: {SITE_NAME}")
print(f"Coordinates: {SITE_LAT}, {SITE_LON}\n")


# =============================================================================
# SECTION 1: DATA PIPELINE -- NASA POWER (climate) + FAO SoilGrids (soil)
# =============================================================================
NASA_POWER_BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
NASA_POWER_PARAMETERS = ["T2M", "T2M_MAX", "T2M_MIN", "RH2M", "PRECTOTCORR",
                          "ALLSKY_SFC_SW_DWN", "WS2M", "GWETTOP"]
SOILGRIDS_BASE_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"
SOILGRIDS_PROPERTIES = ["phh2o", "nitrogen", "soc", "sand", "silt", "clay", "bdod"]
SOILGRIDS_DEPTHS = ["0-5cm", "5-15cm"]


def fetch_nasa_power(max_retries=3):
    print("SECTION 1a: Fetching NASA POWER climate data (1981-today)...")
    params = {
        "parameters": ",".join(NASA_POWER_PARAMETERS), "community": "AG",
        "longitude": SITE_LON, "latitude": SITE_LAT,
        "start": "19810101", "end": datetime.date.today().strftime("%Y%m%d"),
        "format": "JSON",
    }
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(NASA_POWER_BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
            payload = resp.json()
            break
        except requests.RequestException as exc:
            last_error = exc
            print(f"  attempt {attempt}/{max_retries} failed ({exc}), retrying...")
            time.sleep(3 * attempt)
    else:
        raise RuntimeError(f"NASA POWER failed after {max_retries} attempts: {last_error}")

    df = pd.DataFrame(payload["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "date"
    df = df.sort_index().reset_index()
    df = df.replace(-999, pd.NA).replace(-999.0, pd.NA)
    for col in NASA_POWER_PARAMETERS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    print(f"  -> {len(df)} daily records fetched")
    return df


import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)

import rasterio


# ISRIC's SoilGrids conversion factors (raw stored value / d_factor = real
# units). These are a stable, documented part of SoilGrids' technical spec
# (not something ISRIC changes) -- unlike the REST API, which returned
# these dynamically in its response, the WCS raster path doesn't include
# this metadata in-band, so it's hardcoded here from ISRIC's published
# technical documentation.
SOILGRIDS_D_FACTORS = {"phh2o": 10, "nitrogen": 100, "soc": 10,
                        "sand": 10, "silt": 10, "clay": 10, "bdod": 100}


def fetch_soilgrids_rest(max_retries=4):
    """Primary path: SoilGrids REST convenience API. Simple and fast when
    it's up, but ISRIC has documented ongoing instability with this
    specific service (their own site: 'we have decided to temporarily
    pause the service... no estimated timeline for restoration')."""
    params = {"lon": SITE_LON, "lat": SITE_LAT, "property": SOILGRIDS_PROPERTIES,
              "depth": SOILGRIDS_DEPTHS, "value": "mean"}
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(SOILGRIDS_BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()
            rows = []
            for layer in payload["properties"]["layers"]:
                prop_name = layer["name"]
                d_factor = layer.get("unit_measure", {}).get("d_factor", 1)
                for depth_info in layer["depths"]:
                    mean_val = depth_info["values"].get("mean")
                    if mean_val is not None:
                        mean_val = mean_val / d_factor
                    rows.append({"property": prop_name, "depth": depth_info["label"], "value": mean_val})
            return pd.DataFrame(rows)
        except requests.RequestException as exc:
            last_error = exc
            wait = 3 * attempt
            print(f"  [REST] attempt {attempt}/{max_retries} failed ({exc}), retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"SoilGrids REST failed after {max_retries} attempts: {last_error}")


def fetch_soilgrids_wcs(buffer_deg=0.01, max_retries=3):
    """Fallback path: SoilGrids via Web Coverage Service (WCS) -- an OGC
    standard raster-access protocol served from maps.isric.org, a
    DIFFERENT ISRIC subdomain/service from the REST API (rest.isric.org).
    ISRIC's own documentation explicitly recommends WCS as 'a more stable
    way to access SoilGrids' during REST API outages. This fetches a
    small raster tile around the exact site coordinate and reads the
    precise pixel value -- same authoritative underlying SoilGrids data,
    different (currently more reliable) access route."""
    lon_min, lon_max = SITE_LON - buffer_deg, SITE_LON + buffer_deg
    lat_min, lat_max = SITE_LAT - buffer_deg, SITE_LAT + buffer_deg

    rows = []
    for prop in SOILGRIDS_PROPERTIES:
        for depth in SOILGRIDS_DEPTHS:
            coverage_id = f"{prop}_{depth}_mean"
            url = (
                f"https://maps.isric.org/mapserv?map=/map/{prop}.map"
                f"&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage"
                f"&COVERAGEID={coverage_id}&FORMAT=GEOTIFF_INT16"
                f"&SUBSET=long({lon_min},{lon_max})"
                f"&SUBSET=lat({lat_min},{lat_max})"
                f"&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326"
                f"&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326"
            )
            last_error = None
            for attempt in range(1, max_retries + 1):
                try:
                    resp = requests.get(url, timeout=45)
                    resp.raise_for_status()
                    with rasterio.io.MemoryFile(resp.content) as memfile:
                        with memfile.open() as dataset:
                            row_idx, col_idx = dataset.index(SITE_LON, SITE_LAT)
                            raw_value = dataset.read(1)[row_idx, col_idx]
                    d_factor = SOILGRIDS_D_FACTORS[prop]
                    value = float(raw_value) / d_factor
                    rows.append({"property": prop, "depth": depth, "value": value})
                    break
                except Exception as exc:
                    last_error = exc
                    wait = 4 * attempt
                    print(f"  [WCS] {coverage_id} attempt {attempt}/{max_retries} failed ({exc}), retrying in {wait}s...")
                    time.sleep(wait)
            else:
                raise RuntimeError(f"SoilGrids WCS failed for {coverage_id} after {max_retries} attempts: {last_error}")

    return pd.DataFrame(rows)


def fetch_soilgrids():
    """Orchestrator: try the REST API first (simple, fast, works when up),
    fall back to WCS (ISRIC's documented stable alternative) if REST
    fails. Only raises an error if BOTH independent access paths fail --
    at that point it's a genuine, total ISRIC-side outage, not something
    retries or an alternate protocol can work around."""
    print("SECTION 1b: Fetching FAO SoilGrids soil data...")
    print("  Trying REST API (primary path)...")
    try:
        soil_df = fetch_soilgrids_rest()
        print(f"  -> {len(soil_df)} soil property rows fetched via REST")
        return soil_df
    except RuntimeError as rest_error:
        print(f"\n  REST API unavailable: {rest_error}")
        print("  Falling back to WCS (ISRIC's documented stable alternative)...\n")
        try:
            soil_df = fetch_soilgrids_wcs()
            print(f"  -> {len(soil_df)} soil property rows fetched via WCS")
            return soil_df
        except Exception as wcs_error:
            raise RuntimeError(
                f"Both SoilGrids access paths failed.\n"
                f"  REST error: {rest_error}\n"
                f"  WCS error: {wcs_error}\n"
                f"This indicates a genuine, total ISRIC-side outage across both "
                f"services, not a transient issue. Check https://www.isric.org "
                f"for status updates, or try again later."
            )


# ---- Run Section 1: fetch the data ----
climate_df = fetch_nasa_power()
soil_df = fetch_soilgrids()
print("\nCell 1 complete -- climate_df and soil_df ready for Cell 2.")


## Cell 2 — Preprocessing
Turns raw data into the ~52 engineered features and 6 prediction targets.

In [ ]:
"""
AgriQDL — CELL 2: PREPROCESSING
================================================================================
Run AFTER Cell 1 (needs climate_df, soil_df already in memory in this same
Colab session).
"""
import numpy as np
import pandas as pd
import json
import os

# =============================================================================
# SECTION 2: PREPROCESSING -- feature engineering + target generation
# (includes the corrected ewm-based semi-empirical target formulas --
# the cumsum-based version had a real saturation bug, fixed and verified
# during development)
# =============================================================================
def preprocess(climate_df, soil_df):
    print("\nSECTION 2: Preprocessing and feature engineering...")
    df = climate_df.dropna(subset=["T2M", "RH2M", "PRECTOTCORR", "GWETTOP"]).reset_index(drop=True)
    cutoff = pd.Timestamp("1984-01-01")  # solar radiation only available from 1984
    df = df[df["date"] >= cutoff].reset_index(drop=True)
    df = df.set_index("date").asfreq("D").ffill(limit=5)
    doy = df.index.dayofyear
    for col in df.columns:
        if df[col].isna().any():
            seasonal_avg = df[col].groupby(doy).transform("mean")
            df[col] = df[col].fillna(seasonal_avg)
    df = df.reset_index()

    soil_point = soil_df.groupby("property")["value"].mean().to_dict()
    soil_baseline = {
        "ph": soil_point["phh2o"], "nitrogen": soil_point["nitrogen"],
        "phosphorus": 12.0, "potassium": 90.0, "soc": soil_point["soc"],
        "sand": soil_point["sand"], "silt": soil_point["silt"],
        "clay": soil_point["clay"], "bdod": soil_point["bdod"],
    }
    print(f"  Soil baseline (from SoilGrids, this exact site): pH={soil_baseline['ph']:.2f}, "
          f"N={soil_baseline['nitrogen']:.2f} g/kg")

    # --- Real time-series targets ---
    df["target_moisture"] = df["GWETTOP"] * 100.0
    df["target_rainfall_7d"] = df["PRECTOTCORR"].shift(-1).rolling(7, min_periods=7).sum().shift(-6)

    # --- Semi-empirical targets (ewm-based, verified non-saturating) ---
    rain_7d = df["PRECTOTCORR"].rolling(7, min_periods=1).sum()
    temp_norm = (df["T2M"] - df["T2M"].min()) / (df["T2M"].max() - df["T2M"].min() + 1e-9)
    moisture_norm = df["GWETTOP"]
    rain_norm = rain_7d / rain_7d.max()
    rain_norm_c = rain_norm - rain_norm.mean()

    n_driver = 0.35 * temp_norm * moisture_norm - 0.30 * rain_norm_c - 0.05
    df["target_nitrogen"] = soil_baseline["nitrogen"] * (1 + n_driver.ewm(span=21).mean().clip(-0.30, 0.30))

    k_driver = -0.30 * rain_norm_c
    df["target_potassium"] = soil_baseline["potassium"] * (1 + k_driver.ewm(span=21).mean().clip(-0.25, 0.25))

    p_driver = 0.20 * (moisture_norm - moisture_norm.mean())
    df["target_phosphorus"] = soil_baseline["phosphorus"] * (1 + p_driver.ewm(span=21).mean().clip(-0.15, 0.15))

    ph_driver = -0.10 * rain_norm_c
    df["target_ph"] = soil_baseline["ph"] * (1 + ph_driver.ewm(span=14).mean().clip(-0.08, 0.08))

    for col in ["target_nitrogen", "target_potassium", "target_phosphorus", "target_ph"]:
        df[col] = df[col].shift(-7)

    # --- Feature engineering (~52 features) ---
    for name, val in soil_baseline.items():
        df[f"soil_{name}"] = val
    df["gdd"] = ((df["T2M_MAX"] + df["T2M_MIN"]) / 2 - 10.0).clip(lower=0)
    df["soil_moisture_deficit"] = 90.0 - (df["GWETTOP"] * 100.0)
    rm30 = df["PRECTOTCORR"].rolling(30, min_periods=1).mean()
    rs30 = df["PRECTOTCORR"].rolling(30, min_periods=1).std().replace(0, np.nan)
    df["rainfall_anomaly_index"] = ((df["PRECTOTCORR"] - rm30) / rs30).fillna(0)
    for var in ["T2M", "RH2M", "PRECTOTCORR", "GWETTOP"]:
        for w in [7, 14, 30]:
            df[f"{var}_roll_mean_{w}d"] = df[var].rolling(w, min_periods=1).mean()
            df[f"{var}_roll_std_{w}d"] = df[var].rolling(w, min_periods=1).std().fillna(0)
    doy2 = df["date"].dt.dayofyear
    df["season_sin"] = np.sin(2 * np.pi * doy2 / 365.25)
    df["season_cos"] = np.cos(2 * np.pi * doy2 / 365.25)
    for lag in [1, 3, 7]:
        df[f"T2M_lag{lag}"] = df["T2M"].shift(lag)
        df[f"PRECTOTCORR_lag{lag}"] = df["PRECTOTCORR"].shift(lag)

    raw_features = ["T2M", "T2M_MAX", "T2M_MIN", "RH2M", "PRECTOTCORR", "ALLSKY_SFC_SW_DWN", "WS2M", "GWETTOP"]
    feature_cols = (raw_features + [f"soil_{k}" for k in soil_baseline]
                     + ["gdd", "soil_moisture_deficit", "rainfall_anomaly_index"]
                     + [f"{v}_roll_{s}_{w}d" for v in ["T2M","RH2M","PRECTOTCORR","GWETTOP"] for s in ["mean","std"] for w in [7,14,30]]
                     + ["season_sin", "season_cos"]
                     + [f"T2M_lag{l}" for l in [1,3,7]] + [f"PRECTOTCORR_lag{l}" for l in [1,3,7]])
    target_cols = ["target_ph", "target_moisture", "target_nitrogen", "target_phosphorus", "target_potassium", "target_rainfall_7d"]

    model_df = df[["date"] + feature_cols + target_cols].dropna().reset_index(drop=True)
    print(f"  Final dataset: {model_df.shape}, {len(feature_cols)} features")

    scaler_params = {}
    scaled_df = model_df.copy()
    for col in feature_cols + target_cols:
        cmin, cmax = model_df[col].min(), model_df[col].max()
        scaler_params[col] = {"min": float(cmin), "max": float(cmax)}
        denom = (cmax - cmin) if (cmax - cmin) != 0 else 1.0
        scaled_df[col] = (model_df[col] - cmin) / denom

    return scaled_df, feature_cols, target_cols, scaler_params, soil_df




# ---- Run Section 2: preprocess and save ----
scaled_df, feature_cols, target_cols, scaler_params, soil_raw = preprocess(climate_df, soil_df)

os.makedirs("data", exist_ok=True)
scaled_df[["date"] + feature_cols].to_csv("data/features_scaled.csv", index=False)
scaled_df[["date"] + target_cols].to_csv("data/targets.csv", index=False)
with open("data/scaler_params.json", "w") as f:
    json.dump(scaler_params, f, indent=2)
soil_raw.to_csv("data/soil_raw.csv", index=False)

n = len(scaled_df)
train_end, val_end = int(n * 0.70), int(n * 0.85)
X = scaled_df[feature_cols].values.astype(np.float32)
Y = scaled_df[target_cols].values.astype(np.float32)
data = {"X_train": X[:train_end], "Y_train": Y[:train_end],
        "X_val": X[train_end:val_end], "Y_val": Y[train_end:val_end],
        "X_test": X[val_end:], "Y_test": Y[val_end:]}
print(f"\nTrain: {len(data['X_train'])}  Val: {len(data['X_val'])}  Test: {len(data['X_test'])}")
print("Cell 2 complete -- features_scaled.csv, targets.csv, scaler_params.json saved.")
print("data dict, feature_cols, target_cols ready for Cell 3+.")


## Cell 3 — Model Definitions
Defines AgriQDL (triple-VQC), CNN-LSTM baseline, and standalone CNN baseline.

In [ ]:
"""
AgriQDL — CELL 3: MODEL DEFINITIONS
================================================================================
Defines AgriQDL (triple-VQC), CNN-LSTM baseline, and standalone CNN baseline.
Run AFTER Cell 2 (needs feature_cols in memory). This cell only DEFINES the
models -- Cell 4 actually trains them.
"""
import math
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np

# =============================================================================
# SECTION 3: MODEL DEFINITIONS -- AgriQDL (triple-VQC), CNN-LSTM, standalone CNN
# =============================================================================
N_QUBITS, N_ENCODING_QUBITS, N_LAYERS = 8, 4, 3
dev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def vqc_circuit(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(N_ENCODING_QUBITS), normalize=True, pad_with=0.0)
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            qml.RY(weights[layer, q, 0], wires=q)
            qml.RZ(weights[layer, q, 1], wires=q)
        for q in range(N_QUBITS):
            qml.CNOT(wires=[q, (q + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]


weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 2)}


class MPSCompressionLayer(nn.Module):
    """Matrix Product State tensor-network compression layer."""
    def __init__(self, input_dim, bond_dim=16, phys_dim=2):
        super().__init__()
        self.n, self.d, self.p = input_dim, bond_dim, phys_dim
        self.init_vector = nn.Parameter(torch.randn(self.d) * 0.1 + 1.0)
        self.cores = nn.Parameter(torch.randn(self.n, self.d, self.p, self.d) * (1.0 / math.sqrt(self.d)))

    def feature_map(self, x):
        angle = (math.pi / 2) * x
        return torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)

    def forward(self, x):
        batch_size = x.shape[0]
        phi = self.feature_map(x)
        v = self.init_vector.unsqueeze(0).expand(batch_size, -1)
        for i in range(self.n):
            M_i = torch.einsum("lpr,bp->blr", self.cores[i], phi[:, i, :])
            v = torch.einsum("bl,blr->br", v, M_i)
        raw_norm = v.norm(dim=-1, keepdim=True)
        z_unit = v / raw_norm.clamp(min=1e-6)
        scale_penalty = (torch.log(raw_norm.squeeze(-1) + 1e-12) ** 2).mean()
        return z_unit, scale_penalty


class AgriQDL(nn.Module):
    """Triple-VQC: three independent shallow (3-layer) VQCs in parallel --
    chosen over a single deeper VQC after ablation testing showed deeper
    circuits perform WORSE (consistent with barren plateau theory)."""
    def __init__(self, input_dim):
        super().__init__()
        self.mps = MPSCompressionLayer(input_dim, bond_dim=16, phys_dim=2)
        self.vqc_a = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.vqc_b = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.vqc_c = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.fc1 = nn.Linear(24, 64)
        self.fc2 = nn.Linear(64, 6)
        self.relu = nn.ReLU()

    def forward(self, x):
        z, scale_penalty = self.mps(x)
        qa, qb, qc = self.vqc_a(z), self.vqc_b(z), self.vqc_c(z)
        q = torch.cat([qa, qb, qc], dim=-1)
        h = self.relu(self.fc1(q))
        return self.fc2(h), scale_penalty

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


class CNNLSTMBaseline(nn.Module):
    """Classical baseline #1: CNN feature extractor + LSTM temporal layer."""
    def __init__(self, input_dim, n_outputs=6, hidden_size=224, conv2_channels=64):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, conv2_channels, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(input_size=conv2_channels, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_outputs)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x)); x = self.pool(x)
        x = self.relu(self.conv2(x)); x = self.pool(x)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


class CNNOnlyBaseline(nn.Module):
    """Classical baseline #2: standalone CNN -- SAME conv feature extractor
    as the CNN-LSTM baseline, but a classical FC head instead of an LSTM.
    This isolates the effect of temporal (LSTM) modeling specifically."""
    def __init__(self, input_dim, n_outputs=6, conv2_channels=64):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, conv2_channels, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        flattened_dim = conv2_channels * (input_dim // 4)
        self.fc1 = nn.Linear(flattened_dim, 128)
        self.fc2 = nn.Linear(128, n_outputs)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x)); x = self.pool(x)
        x = self.relu(self.conv2(x)); x = self.pool(x)
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


TARGET_LABELS = ["pH", "Moisture", "Nitrogen", "Phosphorus", "Potassium", "Rainfall(7d)"]


def compute_metrics(pred, true):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=0))
    mae = np.abs(pred - true).mean(axis=0)
    ss_res = ((true - pred) ** 2).sum(axis=0)
    ss_tot = ((true - true.mean(axis=0)) ** 2).sum(axis=0)
    r2 = 1 - ss_res / (ss_tot + 1e-12)
    return rmse, mae, r2


def rmse_per_target(pred, true):
    return torch.sqrt(((pred - true) ** 2).mean(dim=0))



# ---- Confirm all 3 models can be instantiated correctly ----
_test_agriqdl = AgriQDL(input_dim=len(feature_cols))
_test_cnnlstm = CNNLSTMBaseline(input_dim=len(feature_cols))
_test_cnn = CNNOnlyBaseline(input_dim=len(feature_cols))
print("Cell 3 complete -- all 3 model architectures defined and verified:")
print(f"  AgriQDL (triple-VQC):   {_test_agriqdl.num_parameters():,} parameters")
print(f"  CNN-LSTM baseline:      {_test_cnnlstm.num_parameters():,} parameters")
print(f"  Standalone CNN baseline:{_test_cnn.num_parameters():,} parameters")
del _test_agriqdl, _test_cnnlstm, _test_cnn


## Cell 4 — Training (the long step, ~1.5-2.5 hours)
Trains all 3 models in this one session for a fair comparison.

In [ ]:
"""
AgriQDL — CELL 4: TRAINING (all 3 models, same session, fair timing)
================================================================================
Run AFTER Cell 3. This is the long cell -- roughly 1-1.5 hours total.
Trains AgriQDL, then CNN-LSTM baseline, then standalone CNN baseline, all in
this one session so their training times are fairly comparable.
"""
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# =============================================================================
# SECTION 4: TRAINING -- all three models, same session, fair timing
# =============================================================================
def train_agriqdl(data, feature_cols):
    print("\n" + "=" * 70 + "\nSECTION 4a: Training AgriQDL (triple-VQC)\n" + "=" * 70)
    train_ds = TensorDataset(torch.tensor(data["X_train"]), torch.tensor(data["Y_train"]))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    X_val, Y_val = torch.tensor(data["X_val"]), torch.tensor(data["Y_val"])

    model = AgriQDL(input_dim=len(feature_cols))
    print(f"Parameters: {model.num_parameters():,}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5)
    loss_fn = nn.MSELoss()

    best_val_loss, epochs_no_improve = float("inf"), 0
    total_start = time.time()
    for epoch in range(1, FULL_MAX_EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred, scale_penalty = model(xb)
            task_loss = loss_fn(pred, yb)
            loss = task_loss + SCALE_PENALTY_WEIGHT * scale_penalty
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            train_losses.append(task_loss.item())
        model.eval()
        with torch.no_grad():
            val_pred, _ = model(X_val)
            val_loss = loss_fn(val_pred, Y_val).item()
            val_rmse = rmse_per_target(val_pred, Y_val).numpy()
        scheduler.step(val_loss)
        train_loss = float(np.mean(train_losses))
        if epoch % 5 == 0 or epoch == 1:
            rmse_str = " ".join(f"{l}:{v:.3f}" for l, v in zip(TARGET_LABELS, val_rmse))
            print(f"Epoch {epoch:3d}/{FULL_MAX_EPOCHS} train={train_loss:.5f} val={val_loss:.5f} "
                  f"({time.time()-epoch_start:.1f}s) [{rmse_str}]")
        if np.isnan(train_loss) or np.isnan(val_loss):
            print("!!! NaN detected -- stopping."); break
        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve = val_loss, 0
            torch.save(model.state_dict(), "best_agriqdl_model.pt")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch}."); break
    total_time = time.time() - total_start
    print(f"AgriQDL done. Best val loss: {best_val_loss:.5f}. Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    model.load_state_dict(torch.load("best_agriqdl_model.pt"))
    return model, total_time


def train_classical(model_class, name, data, feature_cols, ckpt_name):
    print("\n" + "=" * 70 + f"\nSECTION 4b/c: Training {name}\n" + "=" * 70)
    train_ds = TensorDataset(torch.tensor(data["X_train"]), torch.tensor(data["Y_train"]))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    X_val, Y_val = torch.tensor(data["X_val"]), torch.tensor(data["Y_val"])

    model = model_class(input_dim=len(feature_cols))
    print(f"Parameters: {model.num_parameters():,}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5)
    loss_fn = nn.MSELoss()

    best_val_loss, epochs_no_improve = float("inf"), 0
    total_start = time.time()
    for epoch in range(1, BASELINE_MAX_EPOCHS + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), Y_val).item()
        scheduler.step(val_loss)
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{BASELINE_MAX_EPOCHS} train={np.mean(train_losses):.5f} val={val_loss:.5f}")
        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve = val_loss, 0
            torch.save(model.state_dict(), ckpt_name)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch}."); break
    total_time = time.time() - total_start
    print(f"{name} done. Best val loss: {best_val_loss:.5f}. Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    model.load_state_dict(torch.load(ckpt_name))
    return model, total_time



# ---- Run Section 4: train all 3 models ----
agriqdl_model, agriqdl_time = train_agriqdl(data, feature_cols)
cnnlstm_model, cnnlstm_time = train_classical(CNNLSTMBaseline, "CNN-LSTM Baseline", data, feature_cols, "best_cnn_lstm.pt")
cnn_model, cnn_time = train_classical(CNNOnlyBaseline, "Standalone CNN Baseline", data, feature_cols, "best_cnn_only.pt")

print("\nCell 4 complete -- all 3 models trained.")
print(f"  AgriQDL:        {agriqdl_time:.0f}s ({agriqdl_time/60:.1f} min)")
print(f"  CNN-LSTM:       {cnnlstm_time:.0f}s ({cnnlstm_time/60:.1f} min)")
print(f"  Standalone CNN: {cnn_time:.0f}s ({cnn_time/60:.1f} min)")


## Cell 5 — Evaluation
Runs all 6 tests (RMSE, MAE, R², parameter count, training time, paired t-tests) and saves results.

In [ ]:
"""
AgriQDL — CELL 5: EVALUATION (6 tests)
================================================================================
Run AFTER Cell 4 (needs agriqdl_model, cnnlstm_model, cnn_model, data,
agriqdl_time, cnnlstm_time, cnn_time all in memory).

Produces:
  TEST 1: RMSE per target, all 3 models
  TEST 2: MAE per target, all 3 models
  TEST 3: R^2 per target, all 3 models
  TEST 4: Parameter count comparison
  TEST 5: Training time comparison (fair -- same session)
  TEST 6: Paired t-tests (3 pairwise comparisons)

Saves final_comparison_results.json -- this is the file to reference when
writing up your results chapter.
"""
import json
import numpy as np
import torch
from scipy import stats

print("=" * 70)
print("SECTION 5: EVALUATION (6 tests)")
print("=" * 70)

X_test, Y_test_np = torch.tensor(data["X_test"]), data["Y_test"]

agriqdl_model.eval()
cnnlstm_model.eval()
cnn_model.eval()
with torch.no_grad():
    agriqdl_pred, _ = agriqdl_model(X_test)
    agriqdl_pred = agriqdl_pred.numpy()
    cnnlstm_pred = cnnlstm_model(X_test).numpy()
    cnn_pred = cnn_model(X_test).numpy()


def compute_metrics(pred, true):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=0))
    mae = np.abs(pred - true).mean(axis=0)
    ss_res = ((true - pred) ** 2).sum(axis=0)
    ss_tot = ((true - true.mean(axis=0)) ** 2).sum(axis=0)
    r2 = 1 - ss_res / (ss_tot + 1e-12)
    return rmse, mae, r2


TARGET_LABELS = ["pH", "Moisture", "Nitrogen", "Phosphorus", "Potassium", "Rainfall(7d)"]

# ---- TESTS 1-3: RMSE, MAE, R2 per target, all 3 models ----
results = {}
for name, pred in [("AgriQDL", agriqdl_pred), ("CNN-LSTM", cnnlstm_pred), ("Standalone CNN", cnn_pred)]:
    rmse, mae, r2 = compute_metrics(pred, Y_test_np)
    results[name] = {"rmse": rmse, "mae": mae, "r2": r2}
    print(f"\n--- {name} ---")
    for i, label in enumerate(TARGET_LABELS):
        print(f"  {label:15s} RMSE={rmse[i]:.4f} MAE={mae[i]:.4f} R2={r2[i]:.4f}")

# ---- TEST 4: Parameter counts ----
param_counts = {"AgriQDL": agriqdl_model.num_parameters(),
                 "CNN-LSTM": cnnlstm_model.num_parameters(),
                 "Standalone CNN": cnn_model.num_parameters()}
print("\n--- TEST 4: Parameter counts ---")
for name, p in param_counts.items():
    print(f"  {name}: {p:,}")
print(f"  AgriQDL reduction vs CNN-LSTM: {(1 - param_counts['AgriQDL']/param_counts['CNN-LSTM'])*100:.1f}%")
print(f"  AgriQDL reduction vs Standalone CNN: {(1 - param_counts['AgriQDL']/param_counts['Standalone CNN'])*100:.1f}%")

# ---- TEST 5: Training time (same session, fair comparison) ----
train_times = {"AgriQDL": agriqdl_time, "CNN-LSTM": cnnlstm_time, "Standalone CNN": cnn_time}
print("\n--- TEST 5: Training time (same session, fair comparison) ---")
for name, t in train_times.items():
    print(f"  {name}: {t:.0f}s ({t/60:.1f} min)")

# ---- TEST 6: Paired t-tests on per-sample squared error ----
print("\n--- TEST 6: Paired t-tests on per-sample squared error ---")
sq_err = {name: ((pred - Y_test_np) ** 2).mean(axis=1)
          for name, pred in [("AgriQDL", agriqdl_pred), ("CNN-LSTM", cnnlstm_pred), ("Standalone CNN", cnn_pred)]}
pairs = [("AgriQDL", "CNN-LSTM"), ("AgriQDL", "Standalone CNN"), ("CNN-LSTM", "Standalone CNN")]
ttest_results = {}
for a, b in pairs:
    t_stat, p_value = stats.ttest_rel(sq_err[b], sq_err[a])
    sig = "significant (p<0.05)" if p_value < 0.05 else "not significant"
    print(f"  {a} vs {b}: t={t_stat:.4f}, p={p_value:.6f} -> {sig}")
    ttest_results[f"{a}_vs_{b}"] = {"t_stat": float(t_stat), "p_value": float(p_value)}

# ---- Save everything ----
final_results = {
    "site": SITE_NAME, "coordinates": {"lat": SITE_LAT, "lon": SITE_LON},
    "metrics": {name: {"rmse": r["rmse"].tolist(), "mae": r["mae"].tolist(), "r2": r["r2"].tolist()}
                for name, r in results.items()},
    "parameter_counts": param_counts, "training_times_sec": train_times,
    "ttest_results": ttest_results,
}
with open("final_comparison_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("\nCell 5 complete -- saved to final_comparison_results.json")
print("This is the file to reference when writing up your Results chapter.")


## Cell 6 — Export Weights for the App
Produces `model_weights.json` — download this and add it to your hosted AgriQDL app folder so "Refresh Now" works for this new plantation.

In [ ]:
"""
AgriQDL — WEIGHT EXPORT (for in-app JavaScript inference)
================================================================================
Exports your trained model's weights, plus the scaler and soil baseline,
into a single JSON file the app can load directly -- enabling real
in-browser predictions with zero Colab dependency for daily refreshes.

Run this ONCE after your final training run (needs best_agriqdl_model.pt,
data/scaler_params.json, data/soil_raw.csv already present, e.g. restored
from Drive). Produces model_weights.json -- download this and place it
alongside index.html in your hosted app folder.

Every number in this file was cross-validated line-by-line against a pure
JavaScript reimplementation (MPS layer, VQC circuit, feature engineering)
before this script was written -- see the accompanying validation notes.
"""
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane"], check=True)

import os
import shutil
import math
import json
import torch
import torch.nn as nn
import pennylane as qml
import pandas as pd

# ---------------------------------------------------------------------------
# Restore required files from Google Drive if this is a fresh session
# (mirrors daily_refresh.py's approach, since this script is also meant to
# work as a standalone entry point, not just right after a training run)
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/AgriQDL"
REQUIRED_FILES = [
    ("best_agriqdl_model.pt", "best_agriqdl_model.pt"),
    ("data/scaler_params.json", "data/scaler_params.json"),
    ("data/soil_raw.csv", "data/soil_raw.csv"),
]

for local_path, drive_relpath in REQUIRED_FILES:
    drive_path = f"{DRIVE_DIR}/{drive_relpath}"
    if os.path.exists(local_path):
        print(f"Found locally: {local_path}")
    elif os.path.exists(drive_path):
        os.makedirs(os.path.dirname(local_path) if os.path.dirname(local_path) else ".", exist_ok=True)
        shutil.copy(drive_path, local_path)
        print(f"Restored from Drive: {local_path}")
    else:
        raise FileNotFoundError(
            f"{local_path} not found locally OR in Drive ({drive_path}). "
            f"Run your full training pipeline at least once first."
        )

print()

N_QUBITS, N_ENCODING_QUBITS, N_LAYERS = 8, 4, 3
dev = qml.device("default.qubit", wires=N_QUBITS)


@qml.qnode(dev, interface="torch")
def vqc_circuit(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(N_ENCODING_QUBITS), normalize=True, pad_with=0.0)
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            qml.RY(weights[layer, q, 0], wires=q)
            qml.RZ(weights[layer, q, 1], wires=q)
        for q in range(N_QUBITS):
            qml.CNOT(wires=[q, (q + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]


weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 2)}


class MPSCompressionLayer(nn.Module):
    def __init__(self, input_dim, bond_dim=16, phys_dim=2):
        super().__init__()
        self.n, self.d, self.p = input_dim, bond_dim, phys_dim
        self.init_vector = nn.Parameter(torch.randn(self.d) * 0.1 + 1.0)
        self.cores = nn.Parameter(torch.randn(self.n, self.d, self.p, self.d) * (1.0 / math.sqrt(self.d)))

    def forward(self, x):
        return x  # not used for export


class AgriQDL(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.mps = MPSCompressionLayer(input_dim, bond_dim=16, phys_dim=2)
        self.vqc_a = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.vqc_b = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.vqc_c = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        self.fc1 = nn.Linear(24, 64)
        self.fc2 = nn.Linear(64, 6)


def export_weights():
    with open("data/scaler_params.json") as f:
        scaler_params = json.load(f)
    feature_cols = [k for k in scaler_params.keys() if not k.startswith("target_")]

    model = AgriQDL(input_dim=len(feature_cols))
    model.load_state_dict(torch.load("best_agriqdl_model.pt", map_location="cpu"))
    model.eval()

    soil_raw = pd.read_csv("data/soil_raw.csv")
    soil_point = soil_raw.groupby("property")["value"].mean().to_dict()
    soil_baseline = {
        "ph": soil_point["phh2o"], "nitrogen": soil_point["nitrogen"],
        "phosphorus": 12.0, "potassium": 90.0, "soc": soil_point["soc"],
        "sand": soil_point["sand"], "silt": soil_point["silt"],
        "clay": soil_point["clay"], "bdod": soil_point["bdod"],
    }

    export = {
        "feature_cols": feature_cols,
        "scaler_params": scaler_params,
        "soil_baseline": soil_baseline,
        "mps": {
            "init_vector": model.mps.init_vector.detach().tolist(),
            "cores": model.mps.cores.detach().tolist(),
            "bond_dim": 16,
            "phys_dim": 2,
        },
        "vqc_a_weights": dict(model.vqc_a.named_parameters())["weights"].detach().tolist(),
        "vqc_b_weights": dict(model.vqc_b.named_parameters())["weights"].detach().tolist(),
        "vqc_c_weights": dict(model.vqc_c.named_parameters())["weights"].detach().tolist(),
        "fc1_weight": model.fc1.weight.detach().tolist(),
        "fc1_bias": model.fc1.bias.detach().tolist(),
        "fc2_weight": model.fc2.weight.detach().tolist(),
        "fc2_bias": model.fc2.bias.detach().tolist(),
    }

    with open("model_weights.json", "w") as f:
        json.dump(export, f)

    import os
    size_kb = os.path.getsize("model_weights.json") / 1024
    print(f"Exported model_weights.json ({size_kb:.0f} KB)")
    print(f"Feature count: {len(feature_cols)}")
    print("Download this file and place it alongside index.html in your hosted app.")


if __name__ == "__main__":
    export_weights()
